# Building plugin caches from source — validation run for datafusion-bio-functions #217

Rebuilds every vepyr plugin cache **from its raw source**, one plugin at a time, against
the head of [datafusion-bio-functions#217](https://github.com/biodatageeks/datafusion-bio-functions/pull/217).

This is the real-chromosome spot-check both reviews on #217 asked for and that the
unit tests cannot stand in for. It exercises, on real data:

| #217 change | Exercised by |
|---|---|
| `ProviderKind::Vcf` wiring | **clinvar**, **spliceai** (`provider = "vcf"`) |
| Sorted-retry when the tier join loses row order | **clinvar** — sparse enough to land on the hash-build side |
| `assume_unique` + `check_assume_unique_sample` | **cadd**, **spliceai** (`assume_unique = true`) |
| Streaming shard write (the OOM case) | **cadd** chr1-2, **dbnsfp** |
| Keep-first dedup (must NOT be skipped) | **alphamissense** (overlapping UniProts) |
| `ScratchGuard` cleanup | any interrupted build (see the disk-hygiene check) |
| `ProviderKind::Bed` | **not covered** — no BED manifest exists in vepyr-plugins |

## Run order is deliberate

Smallest and newest-code-path first, so a broken pipeline is found in minutes rather
than after an 87 GB download:

1. **clinvar** — 192 MB, VCF provider, sparse → sorted-retry path
2. **alphamissense** — 628 MB, dedup path, has a known-good prior result to diff against
3. **spliceai** — 28.5 GB, VCF provider + `assume_unique`
4. **dbnsfp** — 50.3 GB
5. **cadd** — 87.5 GB, `assume_unique` + SNV/indel merge, the OOM case #196 was built for

## Disk is the binding constraint

Sources total **~168 GB compressed** against **~234 GB free**. CADD alone is 87.5 GB
*compressed* and its per-chrom slices are uncompressed (the manifest declares no
`compression`), so chr1 alone is ~14 GB flat. Hence: **download one plugin, build it,
delete its input, keep only the parquet cache.** Never hold two large sources at once.
Every section below ends with a cleanup cell — do not skip it.

## 0. Setup

Paths, remote, and the per-plugin source matrix.

In [ ]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

DATA = Path("/Users/mwiewior/workspace/data_vepyr")
INPUT_ROOT = DATA / "plugin_input"  # scratch: deleted per plugin after its build
CACHE_ROOT = DATA / "plugin_cache"  # kept: the actual deliverable
VARIATION = DATA / "cache" / "116_GRCh38_merged"  # tiering is inherited from this
PLUGINS_REPO = Path("/tmp/vepyr-plugins")  # checkout of the manifests repo
PLUGINS_REF = "plugin-test-fixes"  # branch holding assume_unique etc.

DRIVE_REMOTE = "gdrive-mw"
DRIVE_FOLDER = "1ZT3g31I0LXepORF_dusy47AuXOnbRlZ_"

INPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
assert VARIATION.is_dir(), f"variation cache missing: {VARIATION}"
print(
    "vepyr:",
    subprocess.run(
        [sys.executable, "-c", "import vepyr;print(vepyr.__file__)"],
        capture_output=True,
        text=True,
    ).stdout.strip(),
)

In [ ]:
# The manifests come from vepyr-plugins, resolved by `git worktree add <ref>` -- so the
# ref must be COMMITTED in this checkout. Editing a .source.toml without committing
# silently rebuilds from the old manifest.
if not PLUGINS_REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/biodatageeks/vepyr-plugins.git",
            str(PLUGINS_REPO),
        ],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(PLUGINS_REPO), "fetch", "origin", PLUGINS_REF], check=True
)
subprocess.run(["git", "-C", str(PLUGINS_REPO), "checkout", PLUGINS_REF], check=True)
head = subprocess.run(
    ["git", "-C", str(PLUGINS_REPO), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
print(f"manifests: {PLUGINS_REPO} @ {PLUGINS_REF} ({head})")
print(sorted(x.name for x in (PLUGINS_REPO / "plugins").iterdir()))

In [ ]:
# Per-plugin source matrix. `prefix` is the contig naming INSIDE the source file --
# tabix regions must match it, not what vepyr calls the contig.
# `compression` is what the manifest's [source.csv] declares: a slice must be written
# in exactly that form or the reader mis-parses it.
PLUGINS = {
    "clinvar": dict(
        files=["clinvar/clinvar.vcf.gz", "clinvar/clinvar.vcf.gz.tbi"],
        main="clinvar.vcf.gz",
        kind="vcf",
        prefix="",
        slice_ext=".vcf.gz",
        size_gb=0.19,
        note="sparse -> expect the assert_start_monotonic sorted-retry to fire",
    ),
    "alphamissense": dict(
        files=[
            "alphamissense/AlphaMissense_hg38.bgz.tsv.gz",
            "alphamissense/AlphaMissense_hg38.bgz.tsv.gz.tbi",
        ],
        main="AlphaMissense_hg38.bgz.tsv.gz",
        kind="tsv_gz",
        prefix="chr",
        slice_ext=".tsv.gz",
        size_gb=0.63,
        note="dedup MUST run (overlapping UniProts); assume_unique stays unset",
    ),
    "spliceai": dict(
        files=[
            "spliceai/spliceai_scores.masked.snv.ensembl_mane.grch38.110.vcf.gz",
            "spliceai/spliceai_scores.masked.snv.ensembl_mane.grch38.110.vcf.gz.tbi",
        ],
        main="spliceai_scores.masked.snv.ensembl_mane.grch38.110.vcf.gz",
        kind="vcf",
        prefix="",
        slice_ext=".vcf.gz",
        size_gb=28.5,
        note="assume_unique=true -> check_assume_unique_sample runs",
    ),
    "dbnsfp": dict(
        files=["dbnsfp/dbNSFP5.3.1a_grch38.gz", "dbnsfp/dbNSFP5.3.1a_grch38.gz.tbi"],
        main="dbNSFP5.3.1a_grch38.gz",
        kind="tsv",
        prefix="",
        slice_ext=".tsv",  # manifest: no compression
        size_gb=50.3,
        note="slices are PLAIN tsv -- manifest declares no compression",
    ),
    "cadd": dict(
        files=[
            "cadd/whole_genome_SNVs.tsv.gz",
            "cadd/whole_genome_SNVs.tsv.gz.tbi",
            "cadd/gnomad.genomes.r4.0.indel.tsv.gz",
            "cadd/gnomad.genomes.r4.0.indel.tsv.gz.tbi",
        ],
        main="whole_genome_SNVs.tsv.gz",
        extra="gnomad.genomes.r4.0.indel.tsv.gz",  # merged per chrom, then gsort'd
        kind="tsv",
        prefix="",
        slice_ext=".tsv",
        size_gb=88.8,
        note="SNV+indel concatenated -> MUST gsort or the shard order contract breaks",
    ),
}
for n, p in PLUGINS.items():
    print(
        f"{n:<14} {p['size_gb']:>6.1f} GB  {p['kind']:<7} prefix={p['prefix']!r:<6} {p['note']}"
    )

In [ ]:
def sh(cmd, **kw):
    """Run a shell command, streaming failure output rather than swallowing it."""
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if r.returncode != 0:
        print(r.stdout[-4000:])
        print(r.stderr[-4000:])
        raise RuntimeError(f"rc={r.returncode}: {cmd}")
    return r.stdout


def free_gb(path=DATA):
    return shutil.disk_usage(path).free / 2**30


def require_space(gb, why=""):
    have = free_gb()
    print(f"free: {have:.1f} GB, need ~{gb:.1f} GB  {why}")
    if have < gb:
        raise RuntimeError(
            f"insufficient disk: {have:.1f} GB free, need {gb:.1f} GB. "
            "Delete a previous plugin's input first."
        )


def download(plugin):
    """Fetch only this plugin's files. Sources are pulled one plugin at a time by design."""
    spec = PLUGINS[plugin]
    dest = INPUT_ROOT / plugin
    dest.mkdir(parents=True, exist_ok=True)
    # 2.5x headroom: the compressed source + its uncompressed per-chrom slices.
    require_space(spec["size_gb"] * 2.5, f"({plugin} source + slices)")
    for rel in spec["files"]:
        sh(
            f'rclone copy "{DRIVE_REMOTE}:{rel}" "{dest}/" '
            f"--drive-root-folder-id {DRIVE_FOLDER} --low-level-retries 50 --progress --stats 30s"
        )
    for rel in spec["files"]:  # rclone can exit 0 having copied nothing
        f = dest / Path(rel).name
        assert f.exists() and f.stat().st_size > 0, f"download silently failed: {f}"
        print(f"  ok {f.name}  {f.stat().st_size / 2**30:.2f} GB")

## 1. Slice + build helpers

One chromosome per `build_plugin_cache` call — never the whole genome at once. Memory is
bounded (~3-4 GB RSS) by #217's streaming write regardless of chromosome size; **time**,
not memory, is the binding constraint.

In [ ]:
def slice_chrom(plugin, chrom):
    """Extract one chromosome from the raw source, in the exact form the manifest expects."""
    spec = PLUGINS[plugin]
    d = INPUT_ROOT / plugin
    src = d / spec["main"]
    region = f"{spec['prefix']}{chrom}"
    out = d / f"{plugin}_chr{chrom}{spec['slice_ext']}"
    if out.exists():
        print(f"  reusing {out.name}")
        return out

    if spec["kind"] == "vcf":
        # -h keeps the header: the VCF provider needs it to type the INFO columns.
        sh(f'tabix -h "{src}" {region} | bgzip -c > "{out}"')
    elif spec["kind"] == "tsv_gz":
        sh(f'tabix "{src}" {region} | gzip -c > "{out}"')
    elif plugin == "cadd":
        # CADD is two tabix'd files sharing one schema. Concatenating two individually
        # sorted files does NOT give a globally sorted file, and the streaming write
        # assumes position-ascending input -- gsort is what makes that true. Without it
        # #217's assert_start_monotonic fires and the sorted-retry path picks it up,
        # which is correct but far slower.
        snv, indel = src, d / spec["extra"]
        raw = d / f"cadd_chr{chrom}_raw.tsv"
        sh(f'{{ tabix "{snv}" {region}; tabix "{indel}" {region}; }} > "{raw}"')
        sh(f'gsort -t $\'\\t\' -k2,2n -S 1G --parallel=4 "{raw}" > "{out}"')
        raw.unlink()
    else:  # plain tsv (dbnsfp)
        sh(f'tabix "{src}" {region} > "{out}"')

    size = out.stat().st_size / 2**30
    assert out.stat().st_size > 0, (
        f"empty slice for {plugin} {region} -- wrong contig prefix?"
    )
    print(f"  {out.name}  {size:.2f} GB")
    return out

In [ ]:
import vepyr


def build_chrom(plugin, chrom, keep_slice=False):
    """Slice -> build -> drop the slice. The parquet shard is what we keep."""
    sl = slice_chrom(plugin, chrom)
    t0 = time.time()
    result = vepyr.build_plugin_cache(
        plugin,
        PLUGINS_REF,
        source_path=str(sl),
        cache_dir=str(VARIATION),
        plugin_cache_root=str(CACHE_ROOT),
        chroms=[str(chrom)],
        plugins_repo=str(PLUGINS_REPO),  # resolves the ref via `git worktree add`
        overwrite=True,
    )
    el = time.time() - t0
    for c, rows, warm, cold in result:
        print(
            f"  {plugin} chr{c}: rows={rows:,} warm={warm:,} cold={cold:,}  [{el / 60:.1f} min]"
        )
    if not keep_slice:
        sl.unlink(missing_ok=True)  # slices are large; the shard is the artifact
    return result

## 2. Validation

`rc=0` is not proof of a correct build. Two invariants are checked on every shard, both
of which #217 touches directly:

- **`(tier, start)` ascending** — `PageDir::resolve_ranges` binary-searches within a tier,
  so an unsorted run silently misses rows that are present. This is exactly what
  `assert_start_monotonic` and the sorted-retry exist to protect.
- **no duplicate probe keys** — for an `assume_unique` plugin a duplicate means the flag is
  wrong and the runtime `HashMap` would keep the *last* row, inverting VEP's first-in-file rule.
  This check is exhaustive, unlike the builder's 2 M-key sample.

In [ ]:
import pyarrow.parquet as pq

MATCH_COLS = {
    "alphamissense": ["protein_variant"],
    "spliceai": ["symbol"],
    "dbnsfp": ["aa_change"],
    "cadd": [],
    "clinvar": [],
}
ASSUME_UNIQUE = {"cadd", "spliceai"}


def validate(plugin, chrom):
    shard = CACHE_ROOT / "plugin" / plugin / f"chr{chrom}.parquet"
    assert shard.exists(), f"no shard written: {shard}"
    t = pq.read_table(shard)
    print(
        f"  {shard.name}: {t.num_rows:,} rows, {t.num_columns} cols, "
        f"{shard.stat().st_size / 2**20:.0f} MB"
    )

    tier = t.column("tier").to_pylist()
    start = t.column("start").to_pylist()
    bad = [
        i
        for i in range(1, len(start))
        if tier[i] == tier[i - 1] and start[i] < start[i - 1]
    ]
    assert not bad, (
        f"{plugin} chr{chrom}: {len(bad)} start regressions within a tier "
        f"(first at row {bad[0]}) -- the shard violates its sorted contract "
        f"and point lookups will silently miss rows"
    )
    assert tier == sorted(tier), (
        f"{plugin} chr{chrom}: tiers not grouped warm-then-cold"
    )

    if plugin in ASSUME_UNIQUE:
        cols = ["start", "allele_string", *MATCH_COLS[plugin]]
        keys = list(zip(*(t.column(c).to_pylist() for c in cols)))
        dupes = len(keys) - len(set(keys))
        assert dupes == 0, (
            f"{plugin} chr{chrom}: {dupes:,} duplicate probe keys -- "
            f"assume_unique=true is WRONG for this source"
        )
        print(f"  assume_unique verified exhaustively over {len(keys):,} keys")
    print(f"  ✓ {plugin} chr{chrom}")

In [ ]:
def cleanup(plugin, keep_source=False):
    """Free the disk for the next plugin. The parquet cache is never touched."""
    d = INPUT_ROOT / plugin
    if keep_source or not d.exists():
        print(f"  keeping {d}")
        return
    size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 2**30
    shutil.rmtree(d)
    print(f"  removed {d} ({size:.1f} GB), free now {free_gb():.1f} GB")


def scratch_check(plugin):
    """#217's ScratchGuard should leave no .tmp behind, even after a failed build."""
    leftovers = list((CACHE_ROOT / "plugin" / plugin).glob("*.tmp"))
    print("  scratch files:", leftovers or "none ✓")
    return leftovers

## 3. Smoke run — chr21 for every plugin

**Run this before any full-genome build.** chr21 is the smallest autosome, so the whole
matrix (5 plugins × download → slice → build → validate) completes quickly and proves the
pipeline end to end. A failure here costs minutes; the same failure found during CADD chr1
costs hours.

Each plugin's input is deleted before the next is downloaded — that ordering is what keeps
the run inside the disk budget.

### 3.1 clinvar

In [ ]:
download("clinvar")
build_chrom("clinvar", 21)
validate("clinvar", 21)
scratch_check("clinvar")
cleanup("clinvar")  # comment out to keep the source for the full-genome run below

### 3.2 alphamissense

In [ ]:
download("alphamissense")
build_chrom("alphamissense", 21)
validate("alphamissense", 21)
scratch_check("alphamissense")
cleanup("alphamissense")  # comment out to keep the source for the full-genome run below

### 3.3 spliceai

In [ ]:
download("spliceai")
build_chrom("spliceai", 21)
validate("spliceai", 21)
scratch_check("spliceai")
cleanup("spliceai")  # comment out to keep the source for the full-genome run below

### 3.4 dbnsfp

In [ ]:
download("dbnsfp")
build_chrom("dbnsfp", 21)
validate("dbnsfp", 21)
scratch_check("dbnsfp")
cleanup("dbnsfp")  # comment out to keep the source for the full-genome run below

### 3.5 cadd

In [ ]:
download("cadd")
build_chrom("cadd", 21)
validate("cadd", 21)
scratch_check("cadd")
cleanup("cadd")  # comment out to keep the source for the full-genome run below

## 4. Full genome, one plugin at a time

Only after §3 is green for all five. Re-downloads the plugin's source, walks chr1..22 + X,
and deletes the source at the end.

**Budget realistically.** CADD chr1 alone runs ~2-4.5 h; the full CADD genome is a
multi-day job. Run one plugin per session and check the shard after each chromosome —
`build_chrom` is idempotent per chromosome (`overwrite=True`), so an interrupted run
resumes by re-running the loop with the finished chromosomes removed from `CHROMS`.

In [ ]:
CHROMS = [str(c) for c in range(1, 23)] + ["X"]


def build_genome(plugin, chroms=CHROMS, stop_on_error=True):
    download(plugin)
    done, failed = [], []
    for c in chroms:
        print(f"\n=== {plugin} chr{c} ({free_gb():.0f} GB free) ===")
        try:
            build_chrom(plugin, c)
            validate(plugin, c)
            done.append(c)
        except Exception as e:
            print(f"  FAILED {plugin} chr{c}: {e}")
            failed.append((c, str(e)))
            if stop_on_error:
                raise
    print(f"\n{plugin}: {len(done)} ok, {len(failed)} failed -> {failed}")
    scratch_check(plugin)
    cleanup(plugin)
    return done, failed

In [ ]:
# One per session. Order matters: smallest first, biggest last.
# build_genome("clinvar")
# build_genome("alphamissense")
# build_genome("spliceai")
# build_genome("dbnsfp")
# build_genome("cadd")

## 5. Result summary

What to report back on #217: per-plugin chromosome coverage, row counts, and — the point of
the exercise — whether anything the PR touches misbehaved on real data.

In [ ]:
rows = []
for plugin in PLUGINS:
    d = CACHE_ROOT / "plugin" / plugin
    if not d.is_dir():
        continue
    shards = sorted(d.glob("chr*.parquet"))
    total = sum(pq.read_metadata(s).num_rows for s in shards)
    size = sum(s.stat().st_size for s in shards) / 2**30
    rows.append((plugin, len(shards), total, size))
    print(f"{plugin:<14} {len(shards):>3} shards  {total:>15,} rows  {size:>6.1f} GB")
print(f"\ncache total: {sum(r[3] for r in rows):.1f} GB in {CACHE_ROOT}")